In [ ]:
import datetime
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import random

import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from keras.models import Sequential
from keras.layers import Dense, LSTM, Input, Flatten, GRU
from keras import backend

from scipy.fft import fft, ifft, fftfreq
import pywt
import warnings
import os
import gc

import math

In [ ]:
def gru_model_ls(hist, x_train, y_train, x_test, y_test, test_len, seed_nr, epochs, cols_list, scaler, plot_yes_no, plot_y_train):
  random.seed(seed_nr)
  np.random.seed(seed_nr)
  tf.random.set_seed(seed_nr)
  model = Sequential()
  model.add(GRU(128, return_sequences=True, input_shape= (x_train.shape[1], x_train.shape[2])))
  model.add(GRU(64, return_sequences=False))
  model.add(Dense(32))
  model.add(Dense(32))
  model.add(Dense(32))
  model.add(Dense(x_train.shape[2]))
  # Compile the model
  model.compile(optimizer='adam', loss='mean_squared_error')
  # Train the model
  model.fit(x_train, y_train, batch_size=1, epochs=epochs)

  predictions_short = model.predict(x_test)
  predictions_short_rescaled = scaler.inverse_transform(predictions_short)
  predictions_short_df = pd.DataFrame(predictions_short_rescaled, columns = cols_list)

  predictions_long = []
  x_test_singular = x_test[0,:]
  x_test_singular = np.reshape(x_test_singular, (1, x_test_singular.shape[0],
                                              x_test_singular.shape[1]))
  predicted_value_long = model.predict(x_test_singular)
  predictions_long.append(predicted_value_long[0].tolist())
  for _ in range(test_len-1):
    x_test_singular = np.append(x_test_singular[0,1:],predicted_value_long, axis=0)
    x_test_singular = np.reshape(x_test_singular, (1, x_test_singular.shape[0],
                                              x_test_singular.shape[1]))
    predicted_value_long = model.predict([x_test_singular])
    predictions_long.append(predicted_value_long[0].tolist())

  predictions_long_rescaled = scaler.inverse_transform(predictions_long)
  predictions_long_df = pd.DataFrame(predictions_long_rescaled, columns = cols_list)


  train = hist[cols_list].iloc[: - test_len]
  valid = hist[cols_list].iloc[-test_len:]
  train = pd.DataFrame(train, columns = cols_list)
  valid = pd.DataFrame(valid, columns = cols_list)

  predictions_short_df.index = valid.index
  predictions_long_df.index = valid.index

  RMSE_short_list = []
  RMSE_long_list = []
  MAE_short_list = []
  MAE_long_list = []
  MAPE_short_list = []
  MAPE_long_list = []
  SDAPE_short_list = []
  SDAPE_long_list = []

  for i in range(len(cols_list)):
    ## RMSE calculation:
    rmse_short_value = np.sqrt(np.mean(((predictions_short_df[predictions_short_df.columns[i]] - valid[valid.columns[i]]) ** 2)))
    rmse_long_value = np.sqrt(np.mean(((predictions_long_df[predictions_long_df.columns[i]] - valid[valid.columns[i]]) ** 2)))
    RMSE_short_list.append(float(rmse_short_value))
    RMSE_long_list.append(float(rmse_long_value))

    ## MAE Calculation:
    mae_short_value = np.mean(np.abs(predictions_short_df[predictions_short_df.columns[i]] - valid[valid.columns[i]]))
    mae_long_value = np.mean(np.abs(predictions_long_df[predictions_long_df.columns[i]] - valid[valid.columns[i]]))
    MAE_short_list.append(float(mae_short_value))
    MAE_long_list.append(float(mae_long_value))

    ## MAPE Calculation:
    mape_short_value = np.mean(np.abs((predictions_short_df[predictions_short_df.columns[i]] - valid[valid.columns[i]])/valid[valid.columns[i]]))
    mape_long_value = np.mean(np.abs((predictions_long_df[predictions_long_df.columns[i]] - valid[valid.columns[i]])/valid[valid.columns[i]]))
    MAPE_short_list.append(float(mape_short_value)*100)
    MAPE_long_list.append(float(mape_long_value)*100)

    ## MAPE Calculation:
    sdape_short_value = np.sqrt(np.mean((np.abs((predictions_short_df[predictions_short_df.columns[i]] - valid[valid.columns[i]])/valid[valid.columns[i]])-mape_short_value)**2))
    sdape_long_value = np.sqrt(np.mean((np.abs((predictions_long_df[predictions_long_df.columns[i]] - valid[valid.columns[i]])/valid[valid.columns[i]])-mape_long_value)**2))
    SDAPE_short_list.append(float(sdape_short_value)*100)
    SDAPE_long_list.append(float(sdape_long_value)*100)





  if plot_yes_no == 'Yes':
    plt.figure(figsize=(16,int(6*len(cols_list))))
    for i in range(len(cols_list)):
      plt.subplot(len(cols_list), 1, i+1)
      plt.xlabel('Date')
      plt.ylabel(cols_list[i])
      plt.plot(train[cols_list[i]], 'b-', label=f'Actual')
      plt.plot(valid[cols_list[i]], 'g-', label=f'Testing')
      plt.plot(predictions_short_df[cols_list[i]], color='r', linestyle='-', label=f'1-Day Prediction')
      plt.plot(predictions_long_df[cols_list[i]], color='firebrick', linestyle='-', label=f'{test_len}-Day Prediction')
      if plot_y_train == "Yes":
          y_train_rescaled = scaler.inverse_transform(y_train)
          plt.plot(train.index[20:], y_train_rescaled[:,i], color='peru', linestyle='-', label='Training')
          # plt.plot(train.index, y_train_rescaled, color='darkkhaki', linestyle='-', label='Training')
      plt.title(valid.columns[i], fontsize=14)
      plt.legend()

  return RMSE_short_list, RMSE_long_list, MAE_short_list, MAE_long_list, MAPE_short_list, MAPE_long_list, SDAPE_short_list, SDAPE_long_list

## Fourier Denoising functions:

In [ ]:
## Padding-based Fourier Transform Denoising (P-FTD), proposed by Song et al. (2021):
def padding_fourier(stock_data, quantile_percent, nr_padding_samples):
  denoised_stock_data = pd.DataFrame()

  for i in range(len(stock_data.columns)):
    xt = stock_data[stock_data.columns[i]].reset_index(drop=True)

    ## Creating the padding:
    X1_bar = np.sqrt(np.mean(xt[:20]))
    X2_bar = np.sqrt(np.mean(xt[-20:]))

    sigma1 = np.sqrt(np.mean(xt[:20]-X1_bar))
    sigma2 = np.sqrt(np.mean(xt[-20:]-X2_bar))

    np.random.seed(i)
    N1_list = np.random.normal(loc=0, scale=sigma1, size=nr_padding_samples)
    N2_list = np.random.normal(loc=0, scale=sigma2, size=nr_padding_samples)

    lower_pad_series = pd.Series(np.flip(N1_list.cumsum())+xt.iloc[0])
    upper_pad_series = pd.Series(N1_list.cumsum()+xt.iloc[-1])

    xt = pd.concat([lower_pad_series, xt, upper_pad_series], ignore_index=True)

    fxt = np.fft.fftn(xt)

    fxt = np.fft.fftshift(fxt)

    quantile_percent_index = int(len(fxt)*(1-quantile_percent))
    one_percent_index = int(len(fxt)*0.01)

    if stock_data.columns[i] == 'Volume':
      threshold_value = one_percent_index
    else:
      threshold_value = quantile_percent_index

    fxt_clean = fxt.copy()
    fxt_clean[:int(len(fxt_clean)/2)-threshold_value] = [0]*(int(len(fxt_clean)/2) - threshold_value)
    fxt_clean[int(len(fxt_clean)/2)+threshold_value:] = [0]*(len(fxt) - int(len(fxt_clean)/2) - threshold_value)

    # print(fxt_clean)
    # plt.plot(fxt)
    # plt.plot(fxt_clean)
    # plt.show()

    fxt_clean = np.fft.ifftshift(fxt_clean)
    xt_clean = np.fft.ifftn(fxt_clean)

    xt_unpadded = xt_clean[nr_padding_samples:-nr_padding_samples]
    denoised_stock_data[stock_data.columns[i]] = xt_unpadded.real

  denoised_stock_data.index = stock_data.index
  return denoised_stock_data

In [ ]:
## Our proposed Linear Drift Denoising (LDD) method:

def linear_drift_denoising(stock_data, quantile_percent):
  denoised_stock_data = pd.DataFrame()

  date_range = np.linspace(0,len(stock_data)-1,len(stock_data))
  for i in range(len(stock_data.columns)):
    xt = stock_data[stock_data.columns[i]]
    lin_drift_slope = (xt.iloc[-1] - xt.iloc[0])/len(xt)
    lin_drift = []
    for j in date_range:
      lin_drift.append(float(xt.iloc[0] + lin_drift_slope * j))
    xt_undrifted = xt - lin_drift
    fxt = np.fft.fft(xt_undrifted)
    if stock_data.columns[i] == 'Volume':
      threshold_value = np.quantile(np.abs(fxt), [0.99])
    else:
      threshold_value = np.quantile(np.abs(fxt), [quantile_percent])
    index = abs(fxt)> threshold_value[0]
    fxt_clean = fxt*index
    xt_clean = np.fft.ifft(fxt_clean)
    xt_clean = xt_clean.real
    xt_clean_redrifted = xt_clean + lin_drift

    denoised_stock_data[stock_data.columns[i]] = xt_clean_redrifted

  denoised_stock_data.index = stock_data.index
  return denoised_stock_data

In [ ]:
## Our proposed Exponential Linear Drift Denoising (Exp-LDD) method:

def lindrift_exponential_denoising(stock_data, exponential_decay_rate):
  denoised_stock_data = pd.DataFrame()

  date_range = np.linspace(0,len(stock_data)-1,len(stock_data))
  for i in range(len(stock_data.columns)):
    xt = stock_data[stock_data.columns[i]]
    lin_drift_slope = (xt.iloc[-1] - xt.iloc[0])/len(xt)
    lin_drift = []
    for j in date_range:
      lin_drift.append(float(xt.iloc[0] + lin_drift_slope * j))
    xt_undrifted = xt - lin_drift
    fxt = np.fft.fft(xt_undrifted)
    fxt_unshifted = fxt.copy()
    fxt = np.fft.fftshift(fxt)
    if stock_data.columns[i]=='Volume':
      exponential_decay_rate_used = 0.8
    else:
      exponential_decay_rate_used = exponential_decay_rate
    exponential_decay_factor = exponential_decay_rate_used**np.abs(np.arange(len(fxt)) - math.floor(len(fxt)/2))
    fxt_clean = fxt*exponential_decay_factor
    fxt_clean = np.fft.ifftshift(fxt_clean)

    plt.plot(fxt_unshifted)
    plt.plot(fxt_clean)
    plt.show()

    xt_clean = np.fft.ifft(fxt_clean)
    xt_clean = xt_clean.real
    xt_clean_redrifted = xt_clean + lin_drift

    denoised_stock_data[stock_data.columns[i]] = xt_clean_redrifted

  denoised_stock_data.index = stock_data.index
  return denoised_stock_data

In [ ]:
## Our proposed Exponential Variable Denoising (Exp-VD) method:

def exponential_variable_denoising(stock_data, quantile_percent, min_max_range, smoothing_range, exponential_decay_rate):
  ## -------------- Creating the variable Min-Max range: ----------------------
  min_val = stock_data.head(min_max_range).min().tolist()
  max_val = stock_data.head(min_max_range).max().tolist()

  min_list_vals = [min_val[:] for _ in range(min_max_range)]
  max_list_vals = [max_val[:] for _ in range(min_max_range)]

  # test_len

  for i in range(len(stock_data)-min_max_range):
    min_val = stock_data.iloc[i:i+min_max_range,:].min().tolist()
    max_val = stock_data.iloc[i:i+min_max_range,:].max().tolist()
    min_list_vals.append(min_val)
    max_list_vals.append(max_val)

  # min_list = pd.DataFrame()
  # max_list = pd.DataFrame()

  min_list = pd.DataFrame(min_list_vals, columns = stock_data.columns.tolist())
  max_list = pd.DataFrame(max_list_vals, columns = stock_data.columns.tolist())

  min_list.index = stock_data.index
  max_list.index = stock_data.index

  ## Setting fixed scaling range for Volume to improve consistency:
  min_list['Volume'] = stock_data['Volume'].min()
  max_list['Volume'] = stock_data['Volume'].max()

  avg_list = (min_list+max_list)/2

  data_scaled = (stock_data - avg_list)/(max_list - min_list)

  ## -------- Smoothing the minmax scaling: --------------
  min_list_smoothed = min_list.iloc[:smoothing_range,:].values.tolist()
  max_list_smoothed = max_list.iloc[:smoothing_range,:].values.tolist()

  for i in range(len(min_list)-int(2*smoothing_range)):
    min_vals_smoothed = min_list.iloc[i:i+int(2*smoothing_range),:].mean().values.tolist()
    min_list_smoothed.append(min_vals_smoothed)

    max_vals_smoothed = max_list.iloc[i:i+int(2*smoothing_range),:].mean().values.tolist()
    max_list_smoothed.append(max_vals_smoothed)

  end_vals_min = min_list.iloc[-smoothing_range:,:].values.tolist()
  min_list_smoothed = min_list_smoothed + end_vals_min

  end_vals_max = max_list.iloc[-smoothing_range:,:].values.tolist()
  max_list_smoothed = max_list_smoothed + end_vals_max

  min_list_smoothed = pd.DataFrame(min_list_smoothed, columns = stock_data.columns.tolist())
  min_list_smoothed.index = min_list.index
  min_list_smoothed

  max_list_smoothed = pd.DataFrame(max_list_smoothed, columns = stock_data.columns.tolist())
  max_list_smoothed.index = max_list.index
  max_list_smoothed

  avg_list_smoothed = (min_list_smoothed+max_list_smoothed)/2

  ## FFT denoising of smoothed curve:
  denoised_stock_data_variable = pd.DataFrame()
  for i in range(len(data_scaled.columns)):
    ## Creating the FFT smoothed curve:
    xt = data_scaled[data_scaled.columns[i]]
    fxt = np.fft.fftn(xt)
    fxt_unshifted = fxt.copy()

    fxt = np.fft.fftshift(fxt)
    # exponential_decay_factor = exponential_decay_rate**np.abs(np.arange(len(fxt)) - len(fxt)/2)
    exponential_decay_factor = exponential_decay_rate**np.abs(np.arange(len(fxt)) - math.floor(len(fxt)/2))
    fxt_clean = fxt*exponential_decay_factor
    fxt_clean = np.fft.ifftshift(fxt_clean)

    plt.plot(fxt_unshifted)
    plt.plot(fxt_clean)
    plt.show()

    xt_clean = np.fft.ifftn(fxt_clean)
    xt_clean = xt_clean.real
    denoised_stock_data_variable[data_scaled.columns[i]] = xt_clean

  denoised_stock_data_variable.index = stock_data.index
  # for i in range(len(data_scaled.columns)):
  #   plt.figure()
  #   plt.plot(data_scaled[data_scaled.columns[i]], 'b-')
  #   plt.plot(denoised_stock_data_variable[data_scaled.columns[i]], 'r-')
  #   plt.title(data_scaled.columns[i])
  #   plt.show()

  ## ------------- Rescaling using the smoothed Min-Max scaling: ----------------

  # denoised_stock_data_rescaled = denoised_stock_data_variable*(max_list - min_list) + avg_list
  denoised_stock_data_rescaled = denoised_stock_data_variable*(max_list_smoothed - min_list_smoothed) + avg_list_smoothed

  return denoised_stock_data_rescaled

In [ ]:
def stock_processor(stock_symbol, start_date, end_date, cols_list, denoising_method, quantile_percent, plot_yes_no):
  test_len = 30
  memory = 20

  stock = yf.Ticker(stock_symbol)
  hist = stock.history(start=start_date, end=end_date, interval='1d')
  # hist.to_csv(f'{stock_symbol}_{start_date}_{end_date}.csv')
  stock_data = hist[cols_list]

  if denoising_method == 'None':
    denoised_stock_data = stock_data
  elif denoising_method == 'P_FTD':  ## The P-FTD method
    nr_padding_samples = 20
    denoised_stock_data = padding_fourier(stock_data.iloc[:-test_len], quantile_percent, nr_padding_samples)
    denoised_stock_data = pd.concat([denoised_stock_data, stock_data.iloc[-test_len:]])
  elif denoising_method == 'LDD':   ## The LDD method
    denoised_stock_data = linear_drift_denoising(stock_data.iloc[:-test_len], quantile_percent)
    denoised_stock_data = pd.concat([denoised_stock_data, stock_data.iloc[-test_len:]])
  elif denoising_method == 'Exp_LDD':    ## The Exp-LDD method
    exponential_decay_rate = 0.95
    denoised_stock_data = lindrift_exponential_denoising(stock_data[:-test_len], exponential_decay_rate)
    denoised_stock_data = pd.concat([denoised_stock_data, stock_data.iloc[-test_len:]])
  elif denoising_method == 'Exp_VD':   ## The Exp-VD method
    min_max_range = 5
    smoothing_range = 5
    exponential_decay_rate = 0.8
    denoised_stock_data = exponential_variable_denoising(stock_data[:-test_len], quantile_percent, min_max_range, smoothing_range, exponential_decay_rate)
    denoised_stock_data = pd.concat([denoised_stock_data, stock_data.iloc[-test_len:]])




  if denoising_method != 'None' and plot_yes_no == 'Yes':
    plt.figure(figsize=(16,int(6*len(cols_list))))
    i = 0
    for col in cols_list:
      i = i+1
      plt.subplot(len(stock_data.columns), 1, i)
      plt.xlabel('Date')
      plt.ylabel(col)
      plt.plot(stock_data[col], color = 'b', linestyle = '-', label = f'Original ({col})')
      plt.plot(denoised_stock_data[col], color = 'firebrick', linestyle = '-', label = f'Denoised ({col})')
      plt.legend()
    plt.show()

  scaler = MinMaxScaler(feature_range=(0,1))
  data_scaled = scaler.fit_transform(denoised_stock_data)

  data_scaled = pd.DataFrame(data_scaled, columns = cols_list) #['Close'])
  data_scaled.index = hist.index
  data_scaled

  x_train = []
  y_train = []
  x_test = []
  y_test = []

  for i in range(len(data_scaled) - test_len - memory):
    x_train.append(data_scaled.iloc[i:i+memory])
    y_train.append(data_scaled.iloc[i+memory])
  for i in range(test_len):
    x_test.append(data_scaled.iloc[-test_len-memory+i:-test_len+i])
    y_test.append(data_scaled.iloc[-test_len+i])

  x_train, y_train = np.array(x_train), np.array(y_train)
  x_test, y_test = np.array(x_test), np.array(y_test)

  return hist, x_train, y_train, x_test, y_test, scaler


In [ ]:
stock_symbol = '^GSPTSE'
start_date = '2024-01-01'
end_date = '2026-01-01'
cols_list = ['Open', 'High', 'Low', 'Close', 'Volume']
denoising_method = 'Exp_LDD'
quantile_percent = 0.95
plot_yes_no = 'Yes'

hist, x_train, y_train, x_test, y_test, scaler = stock_processor(stock_symbol, start_date, end_date, cols_list, denoising_method, quantile_percent, plot_yes_no)

In [ ]:
## Stocks: ['^GSPTSE', 'SLV', 'MSFT', 'TSLA', 'CNQ.TO', 'BHP', 'BMO', 'V', 'L.TO', 'USO']

stock_symbol_list = ['^GSPTSE']

start_date_list = ['2022-01-01', '2022-09-01', '2023-05-01', '2024-01-01']

end_date_list = ['2024-01-01', '2024-09-01', '2025-05-01', '2026-01-01']

cols_list = ['Open', 'High', 'Low', 'Close', 'Volume']

## GRU:

In [ ]:
denoising_method = 'None'
quantile_percent = 0.95

test_len = 30
memory = 20

epochs_lstm = 1
epochs_gru = 1
epochs_dnn = 4

nr_runs = 50

plot_yes_no = 'No'
plot_y_train = 'No'



cols_list_rmse = [item+' (RMSE)' for item in cols_list]
cols_list_mae = [item+' (MAE)' for item in cols_list]
cols_list_mape = [item+' (MAPE)' for item in cols_list]
cols_list_sdape = [item+' (SDAPE)' for item in cols_list]


method_used = 'Orig' if denoising_method=='None' else denoising_method

if os.path.exists(f'Short_GRU_{method_used}_Results.csv') and os.path.exists(f'Long_GRU_{method_used}_Results.csv'):
  print(f'Dataset found ({denoising_method})')
  print('')
  Short_results = pd.read_csv(f'Short_GRU_{method_used}_Results.csv')
  Long_results = pd.read_csv(f'Long_GRU_{method_used}_Results.csv')
  Short_results = Short_results.iloc[:,1:].values.tolist()
  Long_results = Long_results.iloc[:,1:].values.tolist()
else:
  Short_results = []
  Long_results = []

for stock_symbol in stock_symbol_list:
  stock_symbol_processed = stock_symbol.replace('.', '_')
  for i in range(len(start_date_list)):
    start_date = start_date_list[i]
    end_date = end_date_list[i]
    print(f'{stock_symbol} ({start_date} to {end_date})')
    hist, x_train, y_train, x_test, y_test, scaler = stock_processor(stock_symbol, start_date, end_date, cols_list, denoising_method, quantile_percent, plot_yes_no)
    print('')
    print('*'*25, ' GRU ', '*'*25)
    for j in range(nr_runs):
      print('')
      print('-'*50)
      print(f'{stock_symbol} ({start_date} to {end_date})')
      print('Run ', j+1)
      (RMSE_short_list, RMSE_long_list, MAE_short_list, MAE_long_list,
       MAPE_short_list, MAPE_long_list, SDAPE_short_list, SDAPE_long_list) = gru_model_ls(hist, x_train, y_train, x_test, y_test, test_len, j+1, epochs_gru, cols_list, scaler, plot_yes_no, plot_y_train)
      print('RMSE Short Term:', RMSE_short_list)
      print('RMSE Long Term:', RMSE_long_list)
      print('MAE Short Term:', MAE_short_list)
      print('MAE Long Term:', MAE_long_list)
      print('MAPE Short Term:', MAPE_short_list)
      print('MAPE Long Term:', MAPE_long_list)
      print('SDAPE Short Term:', SDAPE_short_list)
      print('SDAPE Long Term:', SDAPE_long_list)
      Short_results.append([stock_symbol_processed, 'GRU', start_date, end_date, j+1] + RMSE_short_list + MAE_short_list + MAPE_short_list + SDAPE_short_list)
      Long_results.append([stock_symbol_processed, 'GRU', start_date, end_date, j+1] + RMSE_long_list + MAE_long_list + MAPE_long_list + SDAPE_long_list)
      backend.clear_session(free_memory = True)
      gc.collect()


Short_results = pd.DataFrame(Short_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list_rmse + cols_list_mae + cols_list_mape + cols_list_sdape)
Long_results = pd.DataFrame(Long_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list_rmse + cols_list_mae + cols_list_mape + cols_list_sdape)
Short_results.to_csv(f'Short_GRU_{method_used}_Results.csv')
Long_results.to_csv(f'Long_GRU_{method_used}_Results.csv')


Long_results

In [ ]:
Short_results[cols_list_mape].median()

In [ ]:
Long_results[cols_list_mape].median()

## Padding-Based Fourier Transform Denoising (P-FTD)

Proposed by Song et al. (2021):

In [ ]:
denoising_method = 'P_FTD'
quantile_percent = 0.95

test_len = 30
memory = 20

epochs_lstm = 1
epochs_gru = 1
epochs_dnn = 4

nr_runs = 50

plot_yes_no = 'No'
plot_y_train = 'No'



cols_list_rmse = [item+' (RMSE)' for item in cols_list]
cols_list_mae = [item+' (MAE)' for item in cols_list]
cols_list_mape = [item+' (MAPE)' for item in cols_list]
cols_list_sdape = [item+' (SDAPE)' for item in cols_list]


method_used = 'Orig' if denoising_method=='None' else denoising_method

if os.path.exists(f'Short_GRU_{method_used}_Results.csv') and os.path.exists(f'Long_GRU_{method_used}_Results.csv'):
  print(f'Dataset found ({denoising_method})')
  print('')
  Short_results = pd.read_csv(f'Short_GRU_{method_used}_Results.csv')
  Long_results = pd.read_csv(f'Long_GRU_{method_used}_Results.csv')
  Short_results = Short_results.iloc[:,1:].values.tolist()
  Long_results = Long_results.iloc[:,1:].values.tolist()
else:
  Short_results = []
  Long_results = []

for stock_symbol in stock_symbol_list:
  stock_symbol_processed = stock_symbol.replace('.', '_')
  for i in range(len(start_date_list)):
    start_date = start_date_list[i]
    end_date = end_date_list[i]
    print(f'{stock_symbol} ({start_date} to {end_date})')
    hist, x_train, y_train, x_test, y_test, scaler = stock_processor(stock_symbol, start_date, end_date, cols_list, denoising_method, quantile_percent, plot_yes_no)
    print('')
    print('*'*25, ' GRU ', '*'*25)
    for j in range(nr_runs):
      print('')
      print('-'*50)
      print(f'{stock_symbol} ({start_date} to {end_date})')
      print('Run ', j+1)
      (RMSE_short_list, RMSE_long_list, MAE_short_list, MAE_long_list,
       MAPE_short_list, MAPE_long_list, SDAPE_short_list, SDAPE_long_list) = gru_model_ls(hist, x_train, y_train, x_test, y_test, test_len, j+1, epochs_gru, cols_list, scaler, plot_yes_no, plot_y_train)
      print('RMSE Short Term:', RMSE_short_list)
      print('RMSE Long Term:', RMSE_long_list)
      print('MAE Short Term:', MAE_short_list)
      print('MAE Long Term:', MAE_long_list)
      print('MAPE Short Term:', MAPE_short_list)
      print('MAPE Long Term:', MAPE_long_list)
      print('SDAPE Short Term:', SDAPE_short_list)
      print('SDAPE Long Term:', SDAPE_long_list)
      Short_results.append([stock_symbol_processed, 'GRU', start_date, end_date, j+1] + RMSE_short_list + MAE_short_list + MAPE_short_list + SDAPE_short_list)
      Long_results.append([stock_symbol_processed, 'GRU', start_date, end_date, j+1] + RMSE_long_list + MAE_long_list + MAPE_long_list + SDAPE_long_list)
      backend.clear_session(free_memory = True)
      gc.collect()


Short_results = pd.DataFrame(Short_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list_rmse + cols_list_mae + cols_list_mape + cols_list_sdape)
Long_results = pd.DataFrame(Long_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list_rmse + cols_list_mae + cols_list_mape + cols_list_sdape)
Short_results.to_csv(f'Short_GRU_{method_used}_Results.csv')
Long_results.to_csv(f'Long_GRU_{method_used}_Results.csv')


Long_results

In [ ]:
Short_results[cols_list_mape].median()

In [ ]:
Long_results[cols_list_mape].median()

## Linear Drift Denoising (LDD):

In [ ]:
denoising_method = 'LDD'
quantile_percent = 0.95

test_len = 30
memory = 20

epochs_lstm = 1
epochs_gru = 1
epochs_dnn = 4

nr_runs = 50

plot_yes_no = 'No'
plot_y_train = 'No'



cols_list_rmse = [item+' (RMSE)' for item in cols_list]
cols_list_mae = [item+' (MAE)' for item in cols_list]
cols_list_mape = [item+' (MAPE)' for item in cols_list]
cols_list_sdape = [item+' (SDAPE)' for item in cols_list]


method_used = 'Orig' if denoising_method=='None' else denoising_method

if os.path.exists(f'Short_GRU_{method_used}_Results.csv') and os.path.exists(f'Long_GRU_{method_used}_Results.csv'):
  print(f'Dataset found ({denoising_method})')
  print('')
  Short_results = pd.read_csv(f'Short_GRU_{method_used}_Results.csv')
  Long_results = pd.read_csv(f'Long_GRU_{method_used}_Results.csv')
  Short_results = Short_results.iloc[:,1:].values.tolist()
  Long_results = Long_results.iloc[:,1:].values.tolist()
else:
  Short_results = []
  Long_results = []

for stock_symbol in stock_symbol_list:
  stock_symbol_processed = stock_symbol.replace('.', '_')
  for i in range(len(start_date_list)):
    start_date = start_date_list[i]
    end_date = end_date_list[i]
    print(f'{stock_symbol} ({start_date} to {end_date})')
    hist, x_train, y_train, x_test, y_test, scaler = stock_processor(stock_symbol, start_date, end_date, cols_list, denoising_method, quantile_percent, plot_yes_no)
    print('')
    print('*'*25, ' GRU ', '*'*25)
    for j in range(nr_runs):
      print('')
      print('-'*50)
      print(f'{stock_symbol} ({start_date} to {end_date})')
      print('Run ', j+1)
      (RMSE_short_list, RMSE_long_list, MAE_short_list, MAE_long_list,
       MAPE_short_list, MAPE_long_list, SDAPE_short_list, SDAPE_long_list) = gru_model_ls(hist, x_train, y_train, x_test, y_test, test_len, j+1, epochs_gru, cols_list, scaler, plot_yes_no, plot_y_train)
      print('RMSE Short Term:', RMSE_short_list)
      print('RMSE Long Term:', RMSE_long_list)
      print('MAE Short Term:', MAE_short_list)
      print('MAE Long Term:', MAE_long_list)
      print('MAPE Short Term:', MAPE_short_list)
      print('MAPE Long Term:', MAPE_long_list)
      print('SDAPE Short Term:', SDAPE_short_list)
      print('SDAPE Long Term:', SDAPE_long_list)
      Short_results.append([stock_symbol_processed, 'GRU', start_date, end_date, j+1] + RMSE_short_list + MAE_short_list + MAPE_short_list + SDAPE_short_list)
      Long_results.append([stock_symbol_processed, 'GRU', start_date, end_date, j+1] + RMSE_long_list + MAE_long_list + MAPE_long_list + SDAPE_long_list)
      backend.clear_session(free_memory = True)
      gc.collect()


Short_results = pd.DataFrame(Short_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list_rmse + cols_list_mae + cols_list_mape + cols_list_sdape)
Long_results = pd.DataFrame(Long_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list_rmse + cols_list_mae + cols_list_mape + cols_list_sdape)
Short_results.to_csv(f'Short_GRU_{method_used}_Results.csv')
Long_results.to_csv(f'Long_GRU_{method_used}_Results.csv')


Long_results

In [ ]:
Short_results[cols_list_mape].median()

In [ ]:
Long_results[cols_list_mape].median()

## Exponential Variable Denoising (Exp-VD):

In [ ]:
denoising_method = 'Exp_VD'
quantile_percent = 0.95

test_len = 30
memory = 20

epochs_lstm = 1
epochs_gru = 1
epochs_dnn = 4

nr_runs = 50

plot_yes_no = 'No'
plot_y_train = 'No'



cols_list_rmse = [item+' (RMSE)' for item in cols_list]
cols_list_mae = [item+' (MAE)' for item in cols_list]
cols_list_mape = [item+' (MAPE)' for item in cols_list]
cols_list_sdape = [item+' (SDAPE)' for item in cols_list]


method_used = 'Orig' if denoising_method=='None' else denoising_method

if os.path.exists(f'Short_GRU_{method_used}_Results.csv') and os.path.exists(f'Long_GRU_{method_used}_Results.csv'):
  print(f'Dataset found ({denoising_method})')
  print('')
  Short_results = pd.read_csv(f'Short_GRU_{method_used}_Results.csv')
  Long_results = pd.read_csv(f'Long_GRU_{method_used}_Results.csv')
  Short_results = Short_results.iloc[:,1:].values.tolist()
  Long_results = Long_results.iloc[:,1:].values.tolist()
else:
  Short_results = []
  Long_results = []

for stock_symbol in stock_symbol_list:
  stock_symbol_processed = stock_symbol.replace('.', '_')
  for i in range(len(start_date_list)):
    start_date = start_date_list[i]
    end_date = end_date_list[i]
    print(f'{stock_symbol} ({start_date} to {end_date})')
    hist, x_train, y_train, x_test, y_test, scaler = stock_processor(stock_symbol, start_date, end_date, cols_list, denoising_method, quantile_percent, plot_yes_no)
    print('')
    print('*'*25, ' GRU ', '*'*25)
    for j in range(nr_runs):
      print('')
      print('-'*50)
      print(f'{stock_symbol} ({start_date} to {end_date})')
      print('Run ', j+1)
      (RMSE_short_list, RMSE_long_list, MAE_short_list, MAE_long_list,
       MAPE_short_list, MAPE_long_list, SDAPE_short_list, SDAPE_long_list) = gru_model_ls(hist, x_train, y_train, x_test, y_test, test_len, j+1, epochs_gru, cols_list, scaler, plot_yes_no, plot_y_train)
      print('RMSE Short Term:', RMSE_short_list)
      print('RMSE Long Term:', RMSE_long_list)
      print('MAE Short Term:', MAE_short_list)
      print('MAE Long Term:', MAE_long_list)
      print('MAPE Short Term:', MAPE_short_list)
      print('MAPE Long Term:', MAPE_long_list)
      print('SDAPE Short Term:', SDAPE_short_list)
      print('SDAPE Long Term:', SDAPE_long_list)
      Short_results.append([stock_symbol_processed, 'GRU', start_date, end_date, j+1] + RMSE_short_list + MAE_short_list + MAPE_short_list + SDAPE_short_list)
      Long_results.append([stock_symbol_processed, 'GRU', start_date, end_date, j+1] + RMSE_long_list + MAE_long_list + MAPE_long_list + SDAPE_long_list)
      backend.clear_session(free_memory = True)
      gc.collect()


Short_results = pd.DataFrame(Short_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list_rmse + cols_list_mae + cols_list_mape + cols_list_sdape)
Long_results = pd.DataFrame(Long_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list_rmse + cols_list_mae + cols_list_mape + cols_list_sdape)
Short_results.to_csv(f'Short_GRU_{method_used}_Results.csv')
Long_results.to_csv(f'Long_GRU_{method_used}_Results.csv')


Long_results

In [ ]:
Short_results[cols_list_mape].median()

In [ ]:
Long_results[cols_list_mape].median()

## Exponential Linear Drift Denoising (Exp-LDD):

In [ ]:
denoising_method = 'Exp_LDD'
quantile_percent = 0.95

test_len = 30
memory = 20

epochs_lstm = 1
epochs_gru = 1
epochs_dnn = 4

nr_runs = 50

plot_yes_no = 'No'
plot_y_train = 'No'



cols_list_rmse = [item+' (RMSE)' for item in cols_list]
cols_list_mae = [item+' (MAE)' for item in cols_list]
cols_list_mape = [item+' (MAPE)' for item in cols_list]
cols_list_sdape = [item+' (SDAPE)' for item in cols_list]


method_used = 'Orig' if denoising_method=='None' else denoising_method

if os.path.exists(f'Short_GRU_{method_used}_Results.csv') and os.path.exists(f'Long_GRU_{method_used}_Results.csv'):
  print(f'Dataset found ({denoising_method})')
  print('')
  Short_results = pd.read_csv(f'Short_GRU_{method_used}_Results.csv')
  Long_results = pd.read_csv(f'Long_GRU_{method_used}_Results.csv')
  Short_results = Short_results.iloc[:,1:].values.tolist()
  Long_results = Long_results.iloc[:,1:].values.tolist()
else:
  Short_results = []
  Long_results = []

for stock_symbol in stock_symbol_list:
  stock_symbol_processed = stock_symbol.replace('.', '_')
  for i in range(len(start_date_list)):
    start_date = start_date_list[i]
    end_date = end_date_list[i]
    print(f'{stock_symbol} ({start_date} to {end_date})')
    hist, x_train, y_train, x_test, y_test, scaler = stock_processor(stock_symbol, start_date, end_date, cols_list, denoising_method, quantile_percent, plot_yes_no)
    print('')
    print('*'*25, ' GRU ', '*'*25)
    for j in range(nr_runs):
      print('')
      print('-'*50)
      print(f'{stock_symbol} ({start_date} to {end_date})')
      print('Run ', j+1)
      (RMSE_short_list, RMSE_long_list, MAE_short_list, MAE_long_list,
       MAPE_short_list, MAPE_long_list, SDAPE_short_list, SDAPE_long_list) = gru_model_ls(hist, x_train, y_train, x_test, y_test, test_len, j+1, epochs_gru, cols_list, scaler, plot_yes_no, plot_y_train)
      print('RMSE Short Term:', RMSE_short_list)
      print('RMSE Long Term:', RMSE_long_list)
      print('MAE Short Term:', MAE_short_list)
      print('MAE Long Term:', MAE_long_list)
      print('MAPE Short Term:', MAPE_short_list)
      print('MAPE Long Term:', MAPE_long_list)
      print('SDAPE Short Term:', SDAPE_short_list)
      print('SDAPE Long Term:', SDAPE_long_list)
      Short_results.append([stock_symbol_processed, 'GRU', start_date, end_date, j+1] + RMSE_short_list + MAE_short_list + MAPE_short_list + SDAPE_short_list)
      Long_results.append([stock_symbol_processed, 'GRU', start_date, end_date, j+1] + RMSE_long_list + MAE_long_list + MAPE_long_list + SDAPE_long_list)
      backend.clear_session(free_memory = True)
      gc.collect()


Short_results = pd.DataFrame(Short_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list_rmse + cols_list_mae + cols_list_mape + cols_list_sdape)
Long_results = pd.DataFrame(Long_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list_rmse + cols_list_mae + cols_list_mape + cols_list_sdape)
Short_results.to_csv(f'Short_GRU_{method_used}_Results.csv')
Long_results.to_csv(f'Long_GRU_{method_used}_Results.csv')

Long_results

In [ ]:
Short_results[cols_list_mape].median()

In [ ]:
Long_results[cols_list_mape].median()